# Research Desk · Fine-tune E5 on Kaggle
Chạy tuần tự với GPU và Internet bật. Upload source ZIP làm Input; không cần API key.
Notebook là quy trình để bạn thực thi; không kèm kết quả full-training đã chạy.
Xem docs/TRAINING.md và docs/KAGGLE.md về dữ liệu, exclusions, OOM, checkpoint và báo cáo.


In [ ]:
from pathlib import Path
import os, zipfile, stat
# Select one GPU before importing torch; two T4 cards do not share one VRAM pool.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
candidates = list(Path('/kaggle/input').rglob('ResearchDesk_Source.zip'))
assert len(candidates) == 1, f'Expected one source ZIP; found {len(candidates)}. Set SOURCE_ZIP manually if needed.'
SOURCE_ZIP = candidates[0]
workspace = Path('/kaggle/working')
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    for info in archive.infolist():
        target = (workspace / info.filename).resolve()
        assert target.is_relative_to(workspace.resolve()), 'Unsafe ZIP path'
        assert not stat.S_ISLNK(info.external_attr >> 16), 'Symlink not allowed'
    archive.extractall(workspace)
PROJECT = workspace / 'research-assist'
assert (PROJECT / 'pyproject.toml').exists()
os.chdir(PROJECT)
print('Project:', PROJECT)


In [ ]:
# Keep Kaggle's CUDA PyTorch. Do not install the local CPU wheel here.
%pip install -e '.[training,dev]'


In [ ]:
import torch, subprocess, sys
assert torch.cuda.is_available(), 'Enable GPU in Kaggle Notebook Settings first.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print('Torch:', torch.__version__)
# This trainer uses one GPU; two T4 cards are not one 32 GB device.


In [ ]:
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], check=True)
run('scripts/download_dataset.py')
run('scripts/prepare_dataset.py')
import json
manifest = json.loads(Path('data/experiments/qasper/manifest.json').read_text())
print(json.dumps(manifest['statistics'], indent=2))


In [ ]:
run('scripts/download_models.py')
print(Path('models/multilingual-e5-small/research_model.json').read_text())


In [ ]:
OUTPUT = 'models/multilingual-e5-research'
run('scripts/train_retriever.py', '--data', 'data/experiments/qasper',
    '--model', 'models/multilingual-e5-small', '--output', OUTPUT,
    '--device', 'cuda', '--epochs', 2, '--batch-size', 16,
    '--max-seq-length', 512, '--seed', 42)
# If OOM: batch-size 8/4 and use a NEW output directory; log changed configuration.


In [ ]:
run('scripts/evaluate_retrieval.py', '--data', 'data/experiments/qasper',
    '--variants', 'configs/benchmark.json', '--device', 'cuda',
    '--output', 'evaluation/retrieval.json')
report = json.loads(Path('evaluation/retrieval.json').read_text())
for variant in report['variants']:
    print(variant['name'], variant['metrics'], 'p95 ms:', variant['latency_p95_ms'])


In [ ]:
checkpoint_zip = workspace / 'research-e5-checkpoint.zip'
model_dir = Path(OUTPUT)
assert (model_dir / 'research_model.json').is_file()
with zipfile.ZipFile(checkpoint_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for file in model_dir.rglob('*'):
        relative = file.relative_to(model_dir)
        if file.is_file() and 'checkpoints' not in relative.parts:
            archive.write(file, str(relative))
experiment_zip = workspace / 'research-e5-experiment.zip'
with zipfile.ZipFile(experiment_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for file in [model_dir/'research_model.json', Path('data/experiments/qasper/manifest.json'),
                 Path('data/datasets/qasper/download.json'), Path('evaluation/retrieval.json')]:
        archive.write(file, str(file))
print('Download from Kaggle Output:', checkpoint_zip, experiment_zip)


## Đưa checkpoint vào ứng dụng
Download hai ZIP qua Kaggle Output. Giải nén checkpoint vào `models/multilingual-e5-research` trên local.
Sửa `.env`: `RA_EMBEDDING_MODEL=models/multilingual-e5-research`, `RA_EMBEDDING_FAMILY=e5`, `RA_LOCAL_DEVICE=cpu`.
Restart backend và Reindex. Nhập `retrieval.json` vào màn hình Thực nghiệm.
Giữ manifest/logs; metric retrieval không là tỷ lệ claim của báo cáo đúng. Dùng thêm gold/human end-to-end evaluation.
